# Database Workflow

This notebook explains the shared database contract used by ERP and MES.

## Communication model
- Client -> ERP: TCP JSON on port 6666
- ERP <-> MES: shared PostgreSQL database (single DB)
- MES -> ERP feedback: writes to ERP tables in the same DB

## Run sequence
1. Database setup (once):
   - PostgreSQL in this environment is configured on port 5000.
   - Create DB: `psql -p 5000 -U postgres -d postgres -c "CREATE DATABASE erp_db;"`
   - Load schema from project root: `psql -p 5000 -U postgres -d erp_db -f Database/shared_schema.sql`
   - The schema creates the dedicated project role `iind_user` with password `IIND20266`.
2. Start ERP full component: `python ERP/run_erp_programs.py --mode all`
3. Optional ERP demo test run: `python ERP/run_erp_programs.py --mode demo`
4. Start MES full component: `python MES/run_mes_programs.py --mode all`
5. Validate DB rows in shared `erp_db` using the notebook query cells

### Running code introduction
The runnable cells at the end of this notebook are designed to be executed in order:
1. Setup and connection validation cell
2. Row-count validation cell

This order ensures database configs are initialized before query helpers are used.

## Implemented Features and Specification Mapping

### ERP specification alignment
- **Section 3.1 (Client Orders)** is supported through persistence in the shared `erp_db` with relational tables for client identity (`Clients`), order headers (`Orders`), and order lines (`Order_Items`).
- The JSON order semantics (client-level metadata + per-item order lines) map directly to normalized tables so downstream planning and dispatch can operate at line level.

### MES specification alignment
- **Section 4.1 (MES Orders)** is supported by `Production_Orders` in the same shared database, plus MES statistics tables to track execution state.
- **Section 4.3 (Statistics)** is supported through `Machine_Stats`, `Tool_Stats`, and `Unloading_Stats` structures.

## Component Structure
- Shared `erp_db`:
  - `Clients`: master data (name/NIF)
  - `Orders`: contractual terms, due date, penalty, state
  - `Order_Items`: order decomposition into producible line items
  - `Simulation_State`: day counter for 60-second day simulation
  - `Production_Feedback`: production/cost feedback register
  - `Production_Orders`: MES work queue and completion state
  - `Machine_Stats`: operating time, utilization proxies, tool changes, pieces processed
  - `Tool_Stats`: per-tool operating time accumulation
  - `Unloading_Stats`: dock-level unloading counts by piece type

## Data Flow Across Components
1. Client submits order JSON to ERP (`TCP 6666`).
2. ERP persists order contract and item lines in shared `erp_db`.
3. ERP planning/dispatch writes production lines into `Production_Orders` in the same DB.
4. MES consumes production lines, runs control logic, and updates statistics tables in the same DB.
5. MES and ERP reporting views query shared tables for status and cost tracking.

In [7]:
import psycopg2

DB_CFG = dict(host='localhost', port='5000', database='erp_db', user='iind_user', password='IIND20266')

def test_conn(cfg, name):
    conn = psycopg2.connect(**cfg)
    conn.close()
    print(f'{name}: connection OK')

test_conn(DB_CFG, 'Shared ERP+MES DB')

Shared ERP+MES DB: connection OK


In [8]:
def count_rows(cfg, table):
    conn = psycopg2.connect(**cfg)
    cur = conn.cursor()
    cur.execute(f'SELECT COUNT(*) FROM {table};')
    n = cur.fetchone()[0]
    cur.close()
    conn.close()
    return n

tables = [
    'Clients',
    'Orders',
    'Order_Items',
    'Simulation_State',
    'Production_Feedback',
    'Production_Orders',
    'Machine_Stats',
    'Tool_Stats',
    'Unloading_Stats',
]

for table_name in tables:
    print(f'{table_name}:', count_rows(DB_CFG, table_name))

Clients: 0
Orders: 0
Order_Items: 0
Simulation_State: 1
Production_Feedback: 0
Production_Orders: 0
Machine_Stats: 0
Tool_Stats: 0
Unloading_Stats: 0
